# Philippine Seismic Activity: Web Scraping PHIVOLCS Data

## Import Libraries

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import urllib3

# disable ssl warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

## Main scraping function

In [18]:
def scrape_phivolcs_earthquakes(start_year=2014, end_year=2025):
    all_earthquakes = []
    
    months = [
        "January", "February", "March", "April", "May", "June",
        "July", "August", "September", "October", "November", "December"
    ]
    
    # loop through each year and month
    for year in range(start_year, end_year + 1):
        print(f"scraping year {year}...")
        
        for month in months:
            try:
                # create url for this month and year
                url = f"https://earthquake.phivolcs.dost.gov.ph/EQLatest-Monthly/{year}/{year}_{month}.html"
                print(f"  scraping {month} {year}")
                
                # get the webpage
                response = requests.get(url, verify=False)
                
                # skip if page doesn't exist
                if response.status_code == 404:
                    print(f"    no data for {month} {year}")
                    continue
                    
                response.raise_for_status()
                
                # parse the html content
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # find all tables on the page
                tables = soup.find_all('table')
                earthquake_table = None
                
                # look for the table that contains earthquake data
                for table in tables:
                    if 'latitude' in table.text.lower() and 'longitude' in table.text.lower():
                        earthquake_table = table
                        break
                
                # skip if no earthquake table found
                if not earthquake_table:
                    print(f"    no earthquake table found for {month} {year}")
                    continue
                
                # get all rows except the header row
                rows = earthquake_table.find_all('tr')[1:]
                
                earthquake_count = 0
                for row in rows:
                    cols = row.find_all('td')
                    if len(cols) >= 6:
                        # extract earthquake data from columns
                        earthquake = {
                            'date_time': cols[0].get_text(strip=True),
                            'latitude': cols[1].get_text(strip=True),
                            'longitude': cols[2].get_text(strip=True),
                            'depth_km': cols[3].get_text(strip=True),
                            'magnitude': cols[4].get_text(strip=True),
                            'location': cols[5].get_text(strip=True),
                            'year': year,
                            'month': month
                        }
                        all_earthquakes.append(earthquake)
                        earthquake_count += 1
                
                print(f"    found {earthquake_count} earthquakes")
                
                time.sleep(0.5)
                
            except Exception as e:
                print(f"    error: {e}")
                continue
    
    print("scraping complete!")
    return pd.DataFrame(all_earthquakes)

In [20]:
df = scrape_phivolcs_earthquakes(2018, 2025)
print(f"earthquakes collected: {len(df)}")

scraping year 2018...
  scraping January 2018
    found 375 earthquakes
  scraping February 2018
    found 367 earthquakes
  scraping March 2018
    found 370 earthquakes
  scraping April 2018
    found 467 earthquakes
  scraping May 2018
    found 599 earthquakes
  scraping June 2018
    found 540 earthquakes
  scraping July 2018
    found 508 earthquakes
  scraping August 2018
    found 529 earthquakes
  scraping September 2018
    found 581 earthquakes
  scraping October 2018
    found 708 earthquakes
  scraping November 2018
    found 659 earthquakes
  scraping December 2018
    found 561 earthquakes
scraping year 2019...
  scraping January 2019
    found 621 earthquakes
  scraping February 2019
    found 930 earthquakes
  scraping March 2019
    found 883 earthquakes
  scraping April 2019
    found 1254 earthquakes
  scraping May 2019
    found 976 earthquakes
  scraping June 2019
    found 787 earthquakes
  scraping July 2019
    found 863 earthquakes
  scraping August 2019
    f

In [22]:
df.head()

,date_time,latitude,longitude,depth_km,magnitude,location,year,month
0,January,,,,,,2018,January
1,31 January 2018 - 11:07 PM,13.20,125.48,025,2.8,082m N 29° E of Palapag (Northern Samar),2018,January
2,31 January 2018 - 10:32 PM,11.68,124.26,007,3.7,011 km S 87° W of Kawayan (Biliran),2018,January
3,31 January 2018 - 08:23 PM,13.05,120.56,034,3.5,018 km S 83° W of Santa Cruz (Occidental Mindoro),2018,January
4,31 January 2018 - 06:42 PM,14.11,120.43,122,1.9,021 km N 82° W of Nasugbu (Batangas),2018,January


In [24]:
df.to_csv('ph_earthquakes_raw.csv', index=False)

## Data validation

In [28]:
missing = df.isnull().sum()
print(missing[missing > 0])

Series([], dtype: int64)


In [30]:
print(df.dtypes)

date_time    object
latitude     object
longitude    object
depth_km     object
magnitude    object
location     object
year          int64
month        object
dtype: object


In [32]:
# convert text numbers to actual numbers
df['magnitude'] = pd.to_numeric(df['magnitude'], errors='coerce')
df['depth_km'] = pd.to_numeric(df['depth_km'], errors='coerce')
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

In [34]:
duplicates = df.duplicated().sum()
duplicates

45

In [36]:
# show the duplicate rows to see what's being duplicated
duplicate_rows = df[df.duplicated(keep=False)]
duplicate_rows.head(10)

,date_time,latitude,longitude,depth_km,magnitude,location,year,month
8870,28 April 2019 - 05:08 AM,9.48,126.35,14.0,2.1,029km N 36° E of Cortes (Surigao Del Sur),2019,April
8871,28 April 2019 - 05:08 AM,9.48,126.35,14.0,2.1,029km N 36° E of Cortes (Surigao Del Sur),2019,April
9377,21 April 2019 - 03:44 AM,6.18,125.92,14.0,2.5,025km S 82° E of Don Marcelino (Davao Occidental),2019,April
9378,21 April 2019 - 03:44 AM,6.18,125.92,14.0,2.5,025km S 82° E of Don Marcelino (Davao Occidental),2019,April
10192,21 May 2019 - 02:22 AM,12.20,124.17,8.0,1.7,012km S 41° E of San Vicente (Northern Samar),2019,May
10193,21 May 2019 - 02:22 AM,12.20,124.17,8.0,1.7,012km S 41° E of San Vicente (Northern Samar),2019,May
10638,08 May 2019 - 03:17 AM,9.61,126.62,11.0,2.6,054km S 69° E of General Luna (Surigao Del Norte),2019,May
10639,08 May 2019 - 03:17 AM,9.61,126.62,11.0,2.6,054km S 69° E of General Luna (Surigao Del Norte),2019,May
11369,13 June 2019 - 01:37 AM,10.61,124.62,3.0,2.4,017km N 48° W of Inopacan (Leyte),2019,June
11370,13 June 2019 - 01:37 AM,10.61,124.62,3.0,2.4,017km N 48° W of Inopacan (Leyte),2019,June


In [38]:
df = df.drop_duplicates()

In [40]:
df['magnitude'].describe()

count    107578.000000
mean          2.513594
std           0.699266
min           1.000000
25%           2.000000
50%           2.400000
75%           2.900000
max           7.500000
Name: magnitude, dtype: float64

In [42]:
df['depth_km'].describe()

count    107535.000000
mean         30.158441
std          38.403373
min           0.000000
25%           9.000000
50%          21.000000
75%          33.000000
max        1068.000000
Name: depth_km, dtype: float64

In [46]:
# convert date_time to proper datetime format
df['datetime'] = pd.to_datetime(df['date_time'], format='%d %B %Y - %I:%M %p', errors='coerce')

In [48]:
df.to_csv('ph_earthquakes_cleaned.csv', index=False)

## Earthquake analysis

In [51]:
# count total number of earthquakes collected
total_quakes = len(df)
total_quakes

107585

In [57]:
# show the yearly earthquake counts for trend analysis
yearly_counts = df['year'].value_counts().sort_index()
yearly_counts

year
2018     6264
2019    12977
2020    14045
2021    12053
2022    14333
2023    16628
2024    18142
2025    13143
Name: count, dtype: int64

In [61]:
# find top 10 most affected provinces/regions
def extract_province(location):
    if '(' in location and ')' in location:
        return location.split('(')[-1].split(')')[0]
    return 'unknown'

df['province'] = df['location'].apply(extract_province)
top_provinces = df['province'].value_counts().head(10)
top_provinces

province
Surigao Del Sur       15612
Davao Oriental         7938
Davao Occidental       7717
Surigao Del Norte      7645
Batangas               5764
Occidental Mindoro     5319
Cagayan                3099
Abra                   2845
Eastern Samar          2646
Ilocos Norte           2529
Name: count, dtype: int64

In [63]:
# find the top 5 strongest earthquakes
strongest = df.nlargest(5, 'magnitude')[['magnitude', 'location', 'year']]
strongest

,magnitude,location,year
82685,7.5,334km N 03° W of Itbayat (Batanes),2024
76271,7.4,029km N 74° E of Hinatuan (Surigao Del Sur),2023
5818,7.2,168km S 40° E of Governor Generoso (Davao Orie...,2018
33605,7.1,236km S 67° E of Jose Abad Santos (Davao Occid...,2021
41169,7.1,097km S 66° E of Governor Generoso (Davao Orie...,2021


In [69]:
# categorize earthquakes by risk level
def magnitude_category(mag):
    if mag < 3.0: return 'minor'
    elif mag < 5.0: return 'light' 
    elif mag < 6.0: return 'moderate'
    else: return 'strong'

df['mag_category'] = df['magnitude'].apply(magnitude_category)

depth_by_mag = df.groupby('mag_category')['depth_km'].mean().sort_values()
depth_by_mag

mag_category
minor       28.945040
light       33.806246
moderate    49.015968
strong      77.369863
Name: depth_km, dtype: float64

In [71]:
# check if recent years have unusually high activity
# compare 2023-2025 average to previous years
recent_avg = df[df['year'] >= 2023]['magnitude'].count() / 3
previous_avg = df[df['year'] < 2023]['magnitude'].count() / 5
activity_change = (recent_avg - previous_avg) / previous_avg * 100
activity_change

33.83893404843711

In [73]:
# check if certain months have more earthquakes
monthly_patterns = df['month'].value_counts().sort_index()
monthly_patterns

month
April         9090
August       10449
December     10322
February      7381
January       8521
July          9340
June          9442
March         8897
May           9043
November      7821
October       8575
September     8704
Name: count, dtype: int64

In [75]:
df.to_csv('ph_earthquakes_analyzed.csv', index=False)